In [2]:
from os import getenv
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import pandas as pd

In [4]:
gemini_api_key = getenv("GEMINI_API_KEY")

In [5]:
df = pd.read_csv("truthfulqa_responses.csv", dtype={'start_time_epoch_s': float, 'end_time_epoch_s': float})
df.dropna(subset=['source'], inplace=True)

In [8]:
questions = df['question'].unique().tolist()
len(questions)

1576

In [10]:
from google.genai import types


In [11]:
import math
import pandas as pd
from google import genai

client = genai.Client(api_key=gemini_api_key)

def batched(iterable, size=100):
    """Yield successive `size`‑length chunks from iterable."""
    for idx in range(0, len(iterable), size):
        yield iterable[idx : idx + size]

all_vectors = []
all_questions = []

for batch in batched(questions, size=100):
    resp = client.models.embed_content(
        model="gemini-embedding-001",
        contents=batch,
        config=types.EmbedContentConfig(task_type="CLUSTERING")
    )
    # gemini returns embeddings in the same order as sent
    batch_vectors = [e.values for e in resp.embeddings]

    all_vectors.extend(batch_vectors)
    all_questions.extend(batch)

# build DataFrame
df = pd.DataFrame(all_vectors)
df["question"] = all_questions

df.to_csv("gemini_question_embeddings.csv", index=False)
print(f"Saved {len(df)} rows → gemini_question_embeddings.csv")


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource has been exhausted (e.g. check quota).', 'status': 'RESOURCE_EXHAUSTED'}}

In [12]:
# build DataFrame
df = pd.DataFrame(all_vectors)
df["question"] = all_questions

df.to_csv("gemini_question_embeddings.csv", index=False)
print(f"Saved {len(df)} rows → gemini_question_embeddings.csv")

Saved 1000 rows → gemini_question_embeddings.csv
